In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd

# own code from src/features/build_features.py:
from src.features.build_features import (
    load_data, clean_data, SalaryVsDeptMedian, build_pipeline,
    get_column_names, save_pipeline, load_pipeline,
    TARGET, PIPELINE_VERSION,
)

EXCEL = "../data/raw/02_CortexHR_AI_ML_Dataset.xlsx"

print("Everything imported fine. Pipeline version:", PIPELINE_VERSION)

Everything imported fine. Pipeline version: features-1.0.0


In [ ]:
#  Load the raw data
raw = load_data(EXCEL)
raw.head(3)  

Loaded 2389 rows, 42 columns from: ../data/raw/02_CortexHR_AI_ML_Dataset.xlsx


,row_id,employee_id,observation_date,department_id,role_id,job_level,work_location,work_mode,employment_type,highest_education,...,paid_leave_days_6m,dept_attrition_rate_prior_12m,left_within_12m,voluntary_left_within_12m,source,exit_date,resignation_date,exit_reason,exit_interview_done,fnf_settlement_amount
0,1,EMP-0002,2025-09-30,D08,R047,4,Pune,Remote,Permanent,B.E. Computer Science,...,5,0.2353,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN
1,2,EMP-0004,2025-09-30,D03,R017,3,Erode,Hybrid,Permanent,MCA,...,4,0.0556,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN
2,3,EMP-0011,2025-09-30,D07,R039,1,Kochi,Hybrid,Permanent,M.A. Social Work,...,7,0.0000,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN


In [3]:
df = clean_data(raw)
df.head(3)

Removed 6 duplicate rows. Rows left: 2383


,department_id,role_id,job_level,work_location,work_mode,employment_type,highest_education,tenure_months,total_experience_years,previous_employers_count,...,training_hours_12m,attendance_rate_3m,late_arrivals_3m,avg_overtime_hours_per_month_3m,avg_hours_worked_3m,wfh_share_3m,unpaid_leave_days_6m,paid_leave_days_6m,dept_attrition_rate_prior_12m,left_within_12m
0,D08,R047,4,Pune,Remote,Permanent,B.E. Computer Science,164,30.8,1,...,34.0,0.9219,3,0.00,8.01,0.891,2.0,5,0.2353,0
1,D03,R017,3,Erode,Hybrid,Permanent,MCA,162,17.5,1,...,48.1,0.9688,1,0.00,8.35,0.469,1.0,4,0.0556,0
2,D07,R039,1,Kochi,Hybrid,Permanent,M.A. Social Work,154,12.8,0,...,57.7,0.9453,0,4.33,8.23,0.297,1.0,7,0.0000,0


In [ ]:
X = df.drop(columns=[TARGET])   # features 
y = df[TARGET]                  # the answer 

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,     # 20% test, 80% train
    stratify=y,        
    random_state=42,   
)

print("Train:", len(X_train), "rows | Test:", len(X_test), "rows")
print("Leavers in train:", round(y_train.mean() * 100, 1), "%")
print("Leavers in test :", round(y_test.mean() * 100, 1), "%")

Train: 1906 rows | Test: 477 rows
Leavers in train: 14.6 %
Leavers in test : 14.7 %


In [5]:
# A quick demo of our custom feature-maker:

little_machine = SalaryVsDeptMedian()
little_machine.fit(X_train)                    # learn the medians from TRAIN

few_rows = X_train[["department_id", "annual_fixed_pay"]].head(5)
result = little_machine.transform(few_rows)    # add the new column

result[["department_id", "annual_fixed_pay", "salary_vs_dept_median"]]

,department_id,annual_fixed_pay,salary_vs_dept_median
772,D09,392016,0.805404
2256,D01,732336,0.781103
2160,D06,627948,1.953303
1014,D01,936744,0.999123
2196,D07,791868,1.579175


In [6]:
pipe = build_pipeline()
pipe   # Jupyter draws the steps for us

Pipeline(steps=[('add_salary_ratio', SalaryVsDeptMedian()),
                ('clean_and_scale',
                 ColumnTransformer(transformers=[('numbers',
                                                  Pipeline(steps=[('fill_missing',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['attendance_rate_3m',
                                                   'late_arrivals_3m',
                                                   'avg_overtime_hours_per_month_3m',
                                                   'tenure_months',
                                                   'salary_vs_dept_median',
                                                   'months_since_last_r...
                                                   'wfh_share_3m',
                                                   'unpaid_leave_days_6m',
                                                   'paid_leave_days_6m',
                                                   'dept_attrition_rate_prior_12m']),
                                                 ('texts',
                                                  Pipeline(steps=[('fill_missing',
                                                                   SimpleImputer(fill_value='Missing',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['department_id', 'role_id',
                                                   'work_location', 'work_mode',
                                                   'employment_type',
                                                   'highest_education'])]))])

In [7]:
# Fit ONE time on the train data, then transform both parts:
pipe.fit(X_train, y_train)

X_train_ready = pipe.transform(X_train)
X_test_ready  = pipe.transform(X_test)

columns = get_column_names(pipe)

print("Train features:", X_train_ready.shape)
print("Test features :", X_test_ready.shape)
print("Number of final columns:", len(columns))
print("Missing values:", int(np.isnan(X_train_ready).sum()), "(must be 0)")
print("\nFirst 10 column names:", columns[:10])

Train features: (1906, 127)
Test features : (477, 127)
Number of final columns: 127
Missing values: 0 (must be 0)

First 10 column names: ['numbers__attendance_rate_3m', 'numbers__late_arrivals_3m', 'numbers__avg_overtime_hours_per_month_3m', 'numbers__tenure_months', 'numbers__salary_vs_dept_median', 'numbers__months_since_last_raise', 'numbers__job_level', 'numbers__total_experience_years', 'numbers__previous_employers_count', 'numbers__commute_distance_km']


In [8]:
saved_file = save_pipeline(pipe, X_train, folder="../models")

Pipeline saved to: ..\models\feature_pipeline_features-1.0.0.joblib


In [9]:
# Load it back and CHECK it gives exactly the same numbers:
loaded_pipe, info = load_pipeline(saved_file)

print("Version:", info["pipeline_version"])

same = np.array_equal(pipe.transform(X_test.head(5)),
                      loaded_pipe.transform(X_test.head(5)))
print("Loaded pipeline gives identical output?", same)

Version: features-1.0.0
Loaded pipeline gives identical output? True


In [10]:
new_raw = pd.read_excel(EXCEL, sheet_name="attrition_scoring_2026")
print("New rows:", len(new_raw))

new_clean = clean_data(new_raw)          # same cleaning as before
new_ready = loaded_pipe.transform(new_clean)

print("Features out:", new_ready.shape)
print("Missing values:", int(np.isnan(new_ready).sum()))
pd.DataFrame(new_ready, columns=columns).head(3)

New rows: 430
Removed 0 duplicate rows. Rows left: 430
Features out: (430, 127)
Missing values: 0


,numbers__attendance_rate_3m,numbers__late_arrivals_3m,numbers__avg_overtime_hours_per_month_3m,numbers__tenure_months,numbers__salary_vs_dept_median,numbers__months_since_last_raise,numbers__job_level,numbers__total_experience_years,numbers__previous_employers_count,numbers__commute_distance_km,...,texts__highest_education_LLM,texts__highest_education_M.A. Social Work,texts__highest_education_M.Com,texts__highest_education_M.Sc Statistics,texts__highest_education_M.Tech,texts__highest_education_MBA,texts__highest_education_MBA Finance,texts__highest_education_MBA HR,texts__highest_education_MCA,texts__highest_education_Missing
0,1.067445,-0.257685,-0.400606,5.107008,2.496101,-0.741245,1.318210,5.070373,-0.184753,-0.027103,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.378731,-1.017191,-0.400606,5.033267,-0.008386,-0.741245,0.438788,2.202768,-0.184753,0.621008,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.378731,-0.510854,0.159114,4.738302,-0.699151,-0.741245,-1.320055,1.189404,-1.045728,3.404697,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
sample = X_test.head(10)
out_1 = pipe.transform(sample)
out_2 = pipe.transform(sample)

# Check 1: shape
print("Rows match   :", out_1.shape[0] == len(sample))
print("Columns match:", out_1.shape[1] == len(columns))

# Check 2: no missing values
print("No missing   :", int(np.isnan(out_1).sum()) == 0)

# Check 3: same input -> same output
print("Deterministic:", np.array_equal(out_1, out_2))

Rows match   : True
Columns match: True
No missing   : True
Deterministic: True


In [12]:
import pathlib
out = pathlib.Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)

pd.DataFrame(X_train_ready, columns=columns).to_csv(out / "X_train.csv", index=False)
pd.DataFrame(X_test_ready,  columns=columns).to_csv(out / "X_test.csv",  index=False)
y_train.to_frame().to_csv(out / "y_train.csv", index=False)
y_test.to_frame().to_csv(out / "y_test.csv", index=False)

print("Saved:", sorted(p.name for p in out.glob("*.csv")))

Saved: ['X_test.csv', 'X_train.csv', 'y_test.csv', 'y_train.csv']
